# Lab 3: Fairness

In this lab, you will explore the concept of ["fairness"](https://en.wikipedia.org/wiki/Fairness_(machine_learning)) in machine learning.
To do this, you will look at
alternative ways to examine a classifier's output,
threshold-based plotting schemes,
protected attributes,
and computational [fairness metrics](https://en.wikipedia.org/wiki/Fairness_(machine_learning)#Group_fairness_criteria).

This assignment is intended to be done collaboratively.  You will work as a group to complete the coding and written portions of this assignment.  However, **each group member will submit a copy of the code** via the autograder and **the group will jointly submit one version of the written portion** via Canvas.

In [ ]:
# Import all libraries we will need.
# It is good style to make all your imports in the first cell of the notebook.

import json
import math
import random
import warnings

import matplotlib.pyplot
import numpy
import pandas
import sklearn.datasets
import sklearn.exceptions
import sklearn.linear_model
import sklearn.metrics

# Set a global seed in case we need some randomness.
random.seed(146)

# We are going to ignore a specific warning from scikit-learn.
# In real-world code you would confront this warning with better data munging and hyperparameter tweaking,
# but those are not the focus of this course.
warnings.simplefilter("ignore", category = sklearn.exceptions.ConvergenceWarning)

## Part 1: The Data

We will be using the same data as Lab 1.
Specifically, that is the data from the ["Communities and Crime" dataset](https://archive.ics.uci.edu/dataset/183/communities+and+crime)
available at [UC Irvine's Machine Learning Repository](https://archive.ics.uci.edu/datasets).
It includes data about the crime rates in various communities,
including information about the socioeconomic, racial, and policing status in each community.
The data is located in the `communities.csv` file.

As before, let's start with loading and shuffling the data.

In [ ]:
communities_data = pandas.read_csv("communities.csv")
communities_data = communities_data.sample(frac = 1, ignore_index = True, random_state = 146)
communities_data

Since we will be working with specific columns in this lab,
let's take a quick peek at all the columns we are working with.

In [ ]:
communities_data.info(verbose = True)

In addition to the base data,
we are also providing an additional file for this lab: `communities_attributes.csv`.
This file contains just a header and a single row.
This row tells us which of our columns are protected.
The possible values are:
 - 0 -- The column is not protected.
 - 1 -- The column is protected.
 - 2 -- The column is the label (which we have already seen from Lab 1 is `ViolentCrimesPerPop`).
 
Let's load this data.

In [ ]:
communities_attributes = pandas.read_csv("communities_attributes.csv")
communities_attributes

Now that we have our base data,
let's create some functions for handling it.

<h3 style="color: darkorange; font-size: x-large";>★ Task 1.A</h3>

Complete the function below that takes in a dataset and returns the train/test splits for the dataset.

Input:
 - A DataFrame representing the entire dataset.
 - The name of the label column.
 - An optional ratio that dictates how much of the data to use for training (and the rest will be used as test data).

Output:
 - Train Features (DataFrame)
 - Test Features (DataFrame)
 - Train Labels (Series)
 - Test Labels (Series)

In [ ]:
def train_test_split(frame, label_column_name, train_ratio = 0.5):
    """
    Take in information about a dataset, and prepare the data for machine learning.

    Returns:
        Train Features
        Test Features
        Train Labels
        Test Labels
    """

    return NotImplemented, NotImplemented, NotImplemented, NotImplemented

communities_features_train, communities_features_test, communities_labels_train, communities_labels_test = train_test_split(communities_data, 'ViolentCrimesPerPop')
if (communities_features_train is NotImplemented):
    # Use fake data until you implement the above function.
    print("Using fake data!")

    communities_features, communities_labels = sklearn.datasets.make_classification(
            n_samples = len(communities_data), n_features = len(communities_data.columns) - 1, random_state = 146)

    # Convert the fake data from numpy to pandas.
    column_names = list(communities_data.columns)
    column_names.remove('ViolentCrimesPerPop')
    communities_features = pandas.DataFrame(communities_features, columns = column_names)
    communities_labels = pandas.Series(communities_labels).astype(numpy.float64)

    # Just use the same data for train and test.
    communities_features_train = communities_features
    communities_features_test = communities_features
    communities_labels_train = communities_labels
    communities_labels_test = communities_labels

print("Counts:")
print("    Train Features: ", len(communities_features_train))
print("    Train Labels:   ", len(communities_labels_train))
print("    Test Features:  ", len(communities_features_test))
print("    Test Labels:    ", len(communities_labels_test))

display(communities_features_train.head(5))
display(communities_features_test.head(5))

Great.
Note that if you sliced into an existing DataFrame,
then the indexes may not have been reset
(the numbering for your frames may not start at 0).
There is nothing inherently wrong with this,
but some people like to have the index line up with the number of rows so you can do `for in range(len(frame)):` loops the same as lists.
If you want to reset the index, you can do with [DataFrame.reset_index(drop = True)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html).

Now let's create a function to deal with the protected columns.

<h3 style="color: darkorange; font-size: x-large";>★ Task 1.B</h3>

Complete the function below that takes in a frame of data and a frame containing the protected attributes,
and returns a new frame without the protected attributes
(both non-protected and label columns should be included).

See the description of `communities_attributes.csv` for information on what `protected_attributes_frame` will look like.
If the protected attributes frame is `None` or empty, return a copy of the passed in data.

In [ ]:
def remove_protected_attributes(frame, protected_attributes_frame):
    """
    Remove protected attributes from data.
    
    Returns:
        A new frame without the protected attributes.
    """

    return NotImplemented

new_data = remove_protected_attributes(communities_data, communities_attributes)
if (new_data is NotImplemented):
    print("Function not implemented.")
else:
    print("Data without protected columns:")
    display(new_data)

To finish off our data munging functions,
let's create a function to mark off records in our dataset into different groups.

<h3 style="color: darkorange; font-size: x-large";>★ Task 1.C</h3>

Complete the following function which will take in a DataFrame, column name, and optional threshold.
For every row in the frame, there will be one entry (with a matching index)
in the returned list that indicates the row's group membership:
 - $ 0 $ if the value in the group column is less than the threshold.
 - $ 1 $ if the value in the group column is greater than or equal to the threshold.

In [ ]:
def get_group_ids(frame, group_column_name, group_threshold = 0.5):
    """
    Create a list describing group membership based on the passed in group and threshold.

    Returns:
        A list that has one element for each row in the passed in frame. 0 if the element is below the threshold, and 1 otherwise.
    """

    return NotImplemented

group_indexes = get_group_ids(communities_data, 'population', group_threshold = 0.10)
if (group_indexes is NotImplemented):
    print("Function not implemented.")
else:
    print(communities_data['population'][0:10])
    print(group_indexes[0:10])

Now that we have our base data, we will also need to be able to split our data into groups based on the value of a specific attribute.

<h3 style="color: green; font-size: x-large";>♦ Group Break</h3>

Take a moment to discus the data.

Some potential things to discuss with your group:
 - You have already used this data in Lab 1. Do you have any new insights about this data?
 - Are there any other attributes that you think should be protected?
 - Are there any protected attributes that you think should not be protected?

## Part 2: Predicting with Probabilities

For this lab, instead of working with direct binary decisions, we want to get more nuance out of our data.
For each prediction, we don't want to see just 1/0, but we want to see information about how certain the classifier is about its prediction.
There are several ways we can do this.

The most straightforward ways would be to train a [regressor](https://en.wikipedia.org/wiki/Regression_analysis), which outputs continuous values instead of discrete ones.
Then we just have to get a version of the communities data that has continuous labels instead of binary ones.
This is very feasible and that data exists.
However, since we have not covered some aspects of machine learning (like [regularization](https://en.wikipedia.org/wiki/Regularization_(mathematics)))
that is key to making regression models work well, we will use another method that we are already more familiar with.

Another way to get continuous output from our classifier is to use information from the classifier itself.
Some classifiers will directly output class labels, but most don't actually do that directly.
Most classification algorithms actually try to compute some score (usually a probability) for each class,
and then predicts the class with the highest score.
To get access to this information, we can use a capable classifier
and the [predict_proba()](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
method instead of the [predict()](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict)
method we have used in previous labs.
Conveniently, the same [sklearn.linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
that we have already used in Labs 1 and 2 is capable of outputting prediction probabilities.

<h3 style="color: darkorange; font-size: x-large";>★ Task 2.A</h3>

Complete the following function which will take in training data and test features,
and output predictions probabilities for the positive label along with the trained classifier.

Note that [predict_proba()](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
will output a probability for each possible label.
In our case, that means two probabilities.
We only want the probability for the positive label (and we can easily use that to compute the negative label probability).

In [ ]:
def train_and_predict(features_train, labels_train, features_test):
    """
    Train a Logistic Regression model and output prediction probabilities for the test features.

    Returns:
        The prediction probabilities for the positive label on the test features.
        The trained classifier.
    """

    return NotImplemented, NotImplemented

communities_predictions, _ = train_and_predict(communities_features_train, communities_labels_train, communities_features_test)
if (communities_predictions is NotImplemented):
    # Use fake predictions until you implement the above function.
    print("Using fake predictions!")
    communities_predictions = [random.random() for _ in range(len(communities_features_test))]

# Compute some continuous score using the prediction probabilities.
score = sklearn.metrics.mean_squared_error(communities_labels_test, communities_predictions)
print("RMSE on the full dataset: ", math.sqrt(score))

When we predict continuous scores instead of binary values,
then there is a very important question that we need to answer:
at what threshold does this prediction become true.
0.5 may seem like an obvious choice,
but in really choosing a prediction threshold is much more difficult.

One way to visualize the effect of prediction threshold choice is to plot the "Receiver Operating Characteristics" (ROC)
(the name comes from using radar to classify incoming planes) of the classifier.
ROC curves have true positive rate on the y-axis and false positive rate on the x-axis.

<center><img src="roc-curve.png"/></center>
<center style='font-size: small'>Image courtesy of <a href='https://en.wikipedia.org/wiki/File:Roc_curve.svg'>Wikimedia Commons</a></center>

At first, this may seem like ROC doesn't have anything to do with prediction thresholds,
but we can see the connection if we look more closely at the two axes.

Intuitively, you can look at an ROC curve as how well your classifier performs as you move the prediction threshold.
Starting at the origin of the x-axis, both false positive rate and true positive rate are zero.
This means that our classifier is very strict about predicting positive labels,
so strict that the prediction threshold starts at 1.0.
Therefore, nothing is above the prediction threshold and nothing is classified as positive (hence the zero FPR and TPR).
As we move right, the threshold is lowered little-by-little.
As some prediction values start to get above the threshold,
points start to get classified as positive.
These points that are getting classified as positive raise the TPR if they are actually positive (correctly classified),
or FNR if they are actually negative (incorrectly classified).

We can easily graph the ROC using the [sklearn.metrics.RocCurveDisplay class](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html).

In [ ]:
sklearn.metrics.RocCurveDisplay.from_predictions(communities_labels_test, communities_predictions)

Here we can see how our classifier behaves as we shift the prediction threshold.
In general, the more area under the ROC curve, the better.
In fact, there is a metric that just calculates the area under an ROC curve: [Area Under the ROC Curve](https://en.wikipedia.org/wiki/Receiver_operating_characteristic#Area_under_the_curve) (AUC or AuROC for short).

In [ ]:
sklearn.metrics.roc_auc_score(communities_labels_test, communities_predictions)

Note that, as expected, we see the same score from `sklearn.metrics.roc_auc_score()` as in the ROC plot.

Another useful way to visualize the trade-offs between different types of binary classification errors is to plot the "Precision-Recall Curve",
where "recall" is another name for true positive rate and "precision" is
$$
\Pr(Y{=}1 \mid \hat{Y}{=}1)
$$
Precision-recall curves have precision on the y-axis and recall on the x-axis.
Like ROC curves, they give us a way to visualize the trade-off between two different metrics.

Also like ROC, more area under the curve is better and there is a metric for how much area is under the curve: Area Under the Precision-Recall Curve (AuPRC).

In [ ]:
sklearn.metrics.PrecisionRecallDisplay.from_predictions(communities_labels_test, communities_predictions)

These graphs and metrics can be very useful when discussing trade-offs in different models.
However for this lab, we want to see some more fairness-focused metrics.
We can take the concept of varying the prediction threshold that we saw here,
and apply it to fairness concepts.

<h3 style="color: green; font-size: x-large";>♦ Group Break</h3>

Take a moment to discus prediction probabilities and ROC curves.

Some potential things to discuss with your group:
 - What (if anything) would be different if we used a linear regressor instead of Logistic Regression?
 - What would the curve for a perfect classifier look like?
 - What's the intuitive difference between a ROC curve and Precision-Recall curve?

## Part 3: Fairness Metrics

How do we measure fairness in our machine learning models?
As we briefly discussed in Lab 2,
measuring fairness is hard and contextual.
There are at least 14 different [definitions of fairness](https://en.wikipedia.org/wiki/Fairness_(machine_learning)#Group_fairness_criteria),
and it is mathematically impossible to guarantee that all fairness metrics can be satisfied at the same time.
We will look at just a couple of these definitions in this lab.
If you are interested in more definitions/metrics for fairness,
you can refer to this [article](https://medium.com/ibm-data-ai/fairness-explained-definitions-and-metrics-9690f8e0a4ea).
Note that we will be using some slightly relaxed variants of the metrics in this lab
(so when in doubt, follow what the lab says over the linked resources).

Before we start with fairness,
we will need to compute some other metrics that we will use to compute our fairness metrics.

<h3 style="color: darkorange; font-size: x-large";>★ Task 3.A</h3>

Complete the function below that takes in the output from a classifier, true labels, and an optional threshold;
and returns a dict that contains that following information (see example output below for exact keys):
 - [F1 Score](https://en.wikipedia.org/wiki/F-score)
 - [False Positive Rate](https://en.wikipedia.org/wiki/False_positive_rate)
 - [True Positive Rate](https://en.wikipedia.org/wiki/Sensitivity_and_specificity)
 - [Positive Rate (also called prevalence)](https://en.wikipedia.org/wiki/Sensitivity_and_specificity#Confusion_matrix)

Whenever a computation would divide by zero, use a `numpy.nan`.

Hint:
All of these stats can be computed from a confusion matrix.
Therefore, you may find the [sklearn.metrics.confusion_matrix()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) function useful here.

Example Output:
```json
{
    "f1": 0.12,
    "fpr": 0.34,
    "tpr": 0.56,
    "pf": 0.78
}
```

In [ ]:
def get_stats(predictions, labels, threshold = 0.5):
    """
    Get scoring stats for the provided predictions, labels, and threshold.

    Example output:
    {
        "f1": 0.12,
        "fpr": 0.34,
        "tpr": 0.56,
        "pf": 0.78
    }

    Returns:
        A dict with full scoring information (see above).
    """

    return NotImplemented

stats = get_stats(communities_predictions, communities_labels_test)
if (stats is NotImplemented):
    print("Function not implemented.")
else:
    print(json.dumps(stats, indent = 4))

Now with the raw stats computed, we can start to compute some fairness metrics.

One of the standard fairness metrics to start with is [Equalized Odds](https://en.wikipedia.org/wiki/Equalized_odds).
Equalized odds says that a classifier is fair if the TPR and FPR for the protected and unprotected groups is the same.
To turn this criteria into a metric/error,
we can just take the sum of absolute differences between TPR and FPR for each group (A and B):

$ |TPR_A - TPR_B| + |FPR_A - FPR_B| $

So if a classifier is perfectly fair, we will get a zero equalized odds error.

<h3 style="color: darkorange; font-size: x-large";>★ Task 3.B</h3>

Complete the function below that computes the equalized odds error for two different predictions results
(given their output from `get_stats()`).

In [ ]:
def equalized_odds(group_a_stats, group_b_stats):
    """
    Compute the equalized odds error for two different predictions results
    (given their output from `get_stats()`).
    
    Returns:
        The equalized odds error.
    """

    return NotImplemented

fake_results = equalized_odds({'tpr': 1.0, 'fpr': 0.0}, {'tpr': 0.0, 'fpr': 1.0})
expected = 2.0

if (fake_results is NotImplemented):
    fake_results = -1.0

print("Got Equalized Odds value of: %0.3f." % (fake_results))
print("Does this value match the expected value of %0.3f? %s." % (expected, str(math.isclose(fake_results, expected))))

In addition to equalized odds,
we will also want to use another fairness metric called 
[Demographic Parity](https://en.wikipedia.org/wiki/Fairness_(machine_learning)#Definitions_based_on_predicted_outcome) also called "Statistical Parity".
A model can satisfy demographic parity if the protected and unprotected groups have
the same probability of being given the positive label.
Or in other words, if both groups have the same positive rate.
Like with equalized odds, we can turn this criteria into a metric/error by taking the absolute difference between the two groups:

$ |PR_A - PR_B| $

<h3 style="color: darkorange; font-size: x-large";>★ Task 3.B</h3>

Complete the function below that computes the demographic parity error for two different predictions results
(given their output from `get_stats()`).

In [ ]:
def demographic_parity(group_a_stats, group_b_stats):
    """
    Compute the demographic parity error for two different predictions results
    (given their output from `get_stats()`).
    
    Returns:
        The demographic parity error.
    """

    return NotImplemented

fake_results = demographic_parity({'pr': 1.0}, {'pr': 0.0})
expected = 1.0

if (fake_results is NotImplemented):
    fake_results = -1.0

print("Got Demographic Parity (gap) value of: %0.3f." % (fake_results))
print("Does this value match the expected value of %0.3f? %s." % (expected, str(math.isclose(fake_results, expected))))

Now that we have ways to compute our fairness metrics,
let's put together all our different functions into a function that can
split, group, train, predict, and plot our data (that's quite a bit).
There is a lot to the following function,
so make sure to take a careful read.

In [ ]:
# Graph the fairness errors for our dataset over different prediction thresholds.
# Put this code in a function so that we can return early if the functions have not been implemented yet.
def plot_fairness_errors(data,
                         title = 'Fairness vs Thresholds',
                         protected_attributes_frame = None,
                         group_column = 'racepctblack', group_threshold = 0.5,
                         label_column_name = 'ViolentCrimesPerPop', prediction_threshold = 0.5,
                         num_thresholds = 100):
    f1_values = []
    eo_values = []
    dp_values = []
    
    thresholds = [i * (1.0 / num_thresholds) for i in range(num_thresholds)]
    aggregate_stats = {}
                             
    # Split the data, make the classifier, and get predictions.
    features_train, features_test, labels_train, labels_test = train_test_split(data, label_column_name)
    if (features_train is NotImplemented):
        print("Required functions have not been implemented.")
        return
                             
    # Put each row into a group.
    group_ids_test = get_group_ids(features_test, group_column, group_threshold = group_threshold)
    if (features_train is NotImplemented):
        print("Required functions have not been implemented.")
        return

    # Remove any protected attributes.
    if (protected_attributes_frame is not None):
        features_train = remove_protected_attributes(features_train, protected_attributes_frame)
        features_test = remove_protected_attributes(features_test, protected_attributes_frame)
        
    # Train with both groups.
    all_predictions, classifier = train_and_predict(features_train, labels_train, features_test)
    if (all_predictions is NotImplemented):
        print("Required functions have not been implemented.")
        return
        
    # Get group-specific labels and predictions.
    labels_test = labels_test.reset_index(drop = True)
                             
    group_0_labels = [labels_test[i] for i in range(len(labels_test)) if (group_ids_test[i] == 0)]
    group_0_predictions = [all_predictions[i] for i in range(len(all_predictions)) if (group_ids_test[i] == 0)]

    group_1_labels = [labels_test[i] for i in range(len(labels_test)) if (group_ids_test[i] == 1)]
    group_1_predictions = [all_predictions[i] for i in range(len(all_predictions)) if (group_ids_test[i] == 1)]
                             
    # Compute stats for each threshold and group.
    for threshold in thresholds:
        stats = get_stats(all_predictions, labels_test, threshold = threshold)
        if (stats is NotImplemented):
            print("Required functions have not been implemented.")
            return

        # Get stats for each group.
        group_0_stats = get_stats(group_0_predictions, group_0_labels, threshold = threshold)
        group_1_stats = get_stats(group_1_predictions, group_1_labels, threshold = threshold)
        
        eo_value = equalized_odds(group_0_stats, group_1_stats)
        if (eo_value is NotImplemented):
            print("Equalized odds function has not been implemented.")
            return

        dp_value = demographic_parity(group_0_stats, group_1_stats)
        if (dp_value is NotImplemented):
            print("Demographic parity function has not been implemented.")
            return
    
        stats['eo'] = eo_value
        stats['dp'] = dp_value

        collect_stats(aggregate_stats, stats)
    
    matplotlib.pyplot.title(title)
    matplotlib.pyplot.xlabel('Prediction Threshold')
    matplotlib.pyplot.ylabel('Score')
    
    matplotlib.pyplot.plot(thresholds, aggregate_stats['f1'], color = 'red', label = 'Overall F1 (Score)')
    matplotlib.pyplot.plot(thresholds, aggregate_stats['eo'], color = 'blue', linestyle = 'dashed', label = 'Equalized Odds (Error)')
    matplotlib.pyplot.plot(thresholds, aggregate_stats['dp'], color = 'green', linestyle = 'dashdot', label = 'Demographic Parity (Error)')
    matplotlib.pyplot.legend()
    
    matplotlib.pyplot.show()
    compute_aggregate_stats(aggregate_stats)

# Collect stats for aggregation.
def collect_stats(aggregate_stats, stats):
    for (key, value) in stats.items():
        if (key not in aggregate_stats):
            aggregate_stats[key] = []
        aggregate_stats[key].append(value)
        
    return aggregate_stats

# Aggregate and print stats.
def compute_aggregate_stats(aggregate_stats):
    for (key, values) in aggregate_stats.items():
        row = (key, numpy.min(values), numpy.mean(values), numpy.median(values), numpy.max(values))
        print("%3s -- Min: %0.3f, Mean: %0.3f, Median: %0.3f, Max: %0.3f" % row)

Let's start by plotting our classifier's performance with protected attributes.

In [ ]:
# Plot our performance with protected attributes.
plot_fairness_errors(communities_data, title = 'Fairness vs Thresholds (With Protected Attributes)')

Here we can see our error metrics plotted against F1 score.
Remember that F1 is a score and larger is better,
while equalized odds and demographic parity are both errors and smaller is better.

We see general agreement in our fairness metrics,
that our model gets more unfair as the threshold value rises.
But how does our model perform when we do not use protected attributes?

In [ ]:
# Plot our performance without protected attributes.
plot_fairness_errors(communities_data, title = 'Fairness vs Thresholds (Without Protected Attributes)', protected_attributes_frame = communities_attributes)

It can be hard to see in the shape of the graph,
but if you look at the scale of the y-axis,
you can see both error metrics are substantially lower.
Confirm this by looking at the raw stats for each metric.
F1 is slightly lower (which is bad),
but equalized odds and demographic parity are also lower (which is good)!

<h3 style="color: green; font-size: x-large";>♦ Group Break</h3>

Take a moment to discuss fairness metrics and our results.

Some potential things to discuss with your group:
 - Given our graphs above, what prediction threshold should we choose?
 - What differences do you see in the graphs when protected attributes were used and not used?
     - What difference do you see in the numeric stats?
 - Which is the better fairness metric for our problem: equalized odds or demographic parity?
 - Are there any [other fairness metrics](https://towardsdatascience.com/a-tutorial-on-fairness-in-machine-learning-3ff8ba1040cb) that can apply to our problem?
 - Is F1 a good choice to represent predictive performance in our graph above?
     - Is there any other metrics we should include?

## Part 4: Short Response Questions

Please go to Canvas to enter your group's responses to the following questions.

#### Q1

If we do not include any protected attributes in our model, will it be fair?

Justify your answer.

#### Q2

Can we use AuROC as a replacement for examing an ROC curve?

Justify your answer.

#### Q3

After removing the protected attributes from our data, did we achieve a more fair classifier?

Justify your answer.